# SWC analysis test notebook

---
1. Step 0 - Preparing for pipeline run, creating folder, creating report, loading labels file (if exists).
2. Step 1 - Creating parquete labels file. The purpose of this file is to label each SWC neuron (by class criteria) and each synapse (post / pre).
3. Step 2 - SWC file simplification. Removing internal nodes without external leaf.
4. Step 3 - Adding labels to the simplified SWC file.
5. Step 4 - Covnerting simplified SWC file + labels to JSON format for the clumpiness calculation.
6. Step 5 - Dividing each json to multiple sub-jsons (sub-trees) of each internal node.
7. Step 6 - Clumpiness calculation via the find-clumpiness program.
8. Step 7 - Joining the clumpiness results of all neurons into a single unified csv file (one row per node, one column per label combination).
9. Step 8 - Assigning the clumpiness scores to the root and the internal nodes of each neuron's SWC file (simplified / raw tree - based on node id).
10. Step 9 - Saving the interactive topology plot of each neuron as an html file (simplified / raw tree).

In [1]:
# Import from custom-made scripts
from scripts.preprocessing import get_neurons_info, simplify_swc_topology, swc2json
from scripts.processing import generate_internal_subtrees, process_single_clumpiness, compile_wide_clumpiness, assign_unified_clumpiness
from scripts.visualize import plot_neuron_topology, plot_scored_neurons
from scripts.report import write_test_report, time_run_steps, STEP_LABELS
from scripts.helpers import read_json

# Imports - python
import pandas as pd
import polars as pl
import numpy as np
import os

In [2]:
# Pipeline class
class ProcessSWC():
    """
    SWC process steps:
    Step 0 - Preparing for pipeline run, creating folder, creating report, loading labels file (if exists).
    Step 1 - Creating parquete labels file. The purpose of this file is to label each SWC neuron (by class criteria) and each synapse (post / pre).
    Step 2 - SWC file simplification. Removing internal nodes without external leaf.
    Step 3 - Adding labels to the simplified SWC file.
    Step 4 - Covnerting simplified SWC file + labels to JSON format for the clumpiness calculation.
    Step 5 - Dividing each json to multiple sub-jsons (sub-trees) of each internal node.
    Step 6 - Clumpiness calculation via the find-clumpiness program.
    Step 7 - Joining the clumpiness results of all neurons into a single unified csv file (one row per node, one column per label combination).
    Step 8 - Assigning the clumpiness scores to the root and the internal nodes of each neuron's SWC file (simplified / raw tree - based on node id).
    Step 9 - Saving the interactive topology plot of each neuron as an html file (simplified / raw tree).
    """



    #####################################################
    ###################### STEP 0 #######################
    #####################################################
    def __init__(self, config_file: str = "config.json"):
        """
        Initiating process step of the class.
        config_file: str -> path to the `config.json` file.
        """

        config = read_json(path=config_file)
        self.dict_config = {
                            "run_name":        config["run_name"],                                   # string value of the pipeline run name
                            "swc_path":        os.path.join(*config["swc_path"].split(",")),         # absulote path of the raw swc files
                            "labels_path":     os.path.join(*config["labels_path"].split(",")),      # absulote path for the labels containing folder
                            "labels_ftr_path": os.path.join(*config["labels_ftr_path"].split(",")),  # absolute path of the labels ftr input file.
                            "overwrite_info":  bool(config["overwrite_info"]),                       # overwrite files if the `run_name` haven't been changed 
                            "data_limit":      config["data_limit"],                                 # number (int) limit of files to process
                            "n_threads":       int(config["n_threads"]),                             # number of threads to be used in the analysis
                            "n_jobs":          int(config["n_jobs"])                                 # number of jobs to process in parallel
                            } 
                         
        # Reading config for paths
        run_name = self.dict_config["run_name"]
        self.dict_paths = {
                           "0-run_log": os.path.join("output", run_name, "0-run_log"),
                           "1-output_labels": os.path.join("output", run_name, "1-output_labels"),
                           "2-swc_simplified": os.path.join("output", run_name, "2-swc_simplified"),
                           "3-swc_simplified_labeled": os.path.join("output", run_name, "3-swc_simplified_labeled"),
                           "4-json": os.path.join("output", run_name, "4-json"),
                           "5_json_divided": os.path.join("output", run_name, "5_json_divided"),
                           "6-clumpiness_scores": os.path.join("output", run_name, "6-clumpiness_scores"),
                           "7-swc_clumpiness_result": os.path.join("output", run_name, "7-swc_clumpiness_result"),
                           "8-swc_clumpiness": os.path.join("output", run_name, "8-swc_clumpiness"),
                           "9-swc_clumpiness_plots": os.path.join("output", run_name, "9-swc_clumpiness_plots")
                            }

        # Creating the folders defined in `self.dict_paths`
        for dir in self.dict_paths.values():
            os.makedirs(dir, exist_ok=True)



    ###############################
    ########## STEP 1 #############
    ###############################
    def step01_create_labels(self):
        """
        First step of the pipeline after the initiation, creation of a 
        parqute format labels file for each neuron in the folder.
        """

        # labels file path
        self.labels_path = os.path.join(self.dict_paths["1-output_labels"], "swc_labels.parquet")

        if os.path.exists(self.labels_path) == False:
            # Getting a list of the aviable SWC file in the swc input folder
            swc_files = [i.split(".")[0] for i in os.listdir(self.dict_config["swc_path"])]
            
            # Creating parquete file -> only relevent swc file by super-type
            get_neurons_info(parquet_path = self.dict_paths["1-output_labels"],
                             labels_path = self.dict_config["labels_path"],
                             overwrite_parquet = self.dict_config["overwrite_info"],
                             swc_labels_file_path = self.dict_config["labels_ftr_path"])



    ##############################
    ########## STEP 2 ############
    ##############################
    def step02_simplify_swc(self,
                            neuron_id: str | int):
        """
        Simplification step of the original raw SWC neuron file.
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """

        # Loading neuron file
        neuron_path = os.path.join(self.dict_config["swc_path"], f"{neuron_id}.swc")
        neuron_swc = pd.read_csv(neuron_path, 
                                comment='#', 
                                header=None, 
                                sep=r'\s+', 
                                names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])


        path_simplified_Swc = os.path.join(self.dict_paths["2-swc_simplified"], f"{neuron_id}.csv")
        if (self.dict_config["overwrite_info"]) or (os.path.exists(path_simplified_Swc) is False):
            #Simplifing SWC file
            simplify_swc_topology(swc_input = neuron_swc, 
                                swc_name = f"{neuron_id}",
                                output_path = self.dict_paths["2-swc_simplified"],
                                save_csv=True)
    
        

    ###############################
    ########### STEP 3 ############
    ###############################
    def step03_label_joining(self,
                             neuron_id: str | int):
        """
        3rd step in the pipeline - adding labels to the simplified swc file.
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """

        # Loading relevent rows from the labels praquet file
        labels_parquet = pl.scan_parquet(self.labels_path).filter(pl.col("neuron").cast(pl.Utf8) == str(neuron_id)).collect().to_pandas()


        # Attach nodes labels to the simplified SWC files
        swc_labeled = pd.merge(left = pd.read_csv(os.path.join(self.dict_paths["2-swc_simplified"], f"{neuron_id}.csv"), index_col=0), 
                               right = labels_parquet[["node_id", "type"]].drop_duplicates(), 
                               left_on = "node_id", 
                               right_on = "node_id", 
                               how = "left")  

        #save simplified swc file
        save_path = os.path.join(self.dict_paths["3-swc_simplified_labeled"], f"{neuron_id}.csv")
        swc_labeled["type"] = swc_labeled.groupby("node_id")["type"].unique().apply(lambda X : X[0] if len(X) <= 1 else ",".join(X)) # joining labels if more then 2 per node
        swc_labeled = swc_labeled.drop_duplicates(subset=["node_id", "parent"], keep="first")                                        # dropping rows with the same parent+node_id
        
        # Saving output file
        if (self.dict_config["overwrite_info"]) or (os.path.exists(save_path) is False):
            swc_labeled.to_csv(save_path)



    ###############################
    ########### STEP 4 ############
    ###############################
    def step04_json_creation(self,
                             neuron_id:str | int):
        """
        4th step in the pipeline - creating json file from the simplified swc + labels file.
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """

        #labels swc file import
        df_swc_labeled = pd.read_csv(os.path.join(self.dict_paths["3-swc_simplified_labeled"], f"{neuron_id}.csv"), index_col = 0)
        path_json = os.path.join(self.dict_paths["4-json"])

        # JSON conversion function
        swc2json(swc_dataset = df_swc_labeled,
                 neuron_id = neuron_id,
                 save_json = True,
                 save_path = path_json,
                 overwrite = self.dict_config["overwrite_info"])



    ###############################
    ########### STEP 5 ############
    ###############################
    def step05_internal_node_div(self,
                                 neuron_id : int | str):
        """
        5th step of the pipeline. Creating seperate JSON file for each sub-tree whithin the main tree.
        This step is useful for the individual clumpiness scoring of each internal node.
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """

        # Required paths
        path_main_json = os.path.join(self.dict_paths["4-json"], f"{neuron_id}.json")
        path_div_json = self.dict_paths["5_json_divided"]

        # JSON split function
        generate_internal_subtrees(input_json_path = path_main_json, 
                                   neuron_number = neuron_id, 
                                   output_dir = path_div_json,
                                   overwrite = self.dict_config["overwrite_info"])



    ########################################
    ################ STEP 6 ################
    ########################################
    def step06_clumpiness_calculation(self,
                                       neuron_id : int | str):
        """
        6th step, calculating the clumpiness for each json file
        neuron_id: str | int -> neuron id number of the file in the `swc_input` folder.
        """
        
        path_main = os.path.join(self.dict_paths["4-json"], f"{neuron_id}.json")
        path_div = os.path.join(self.dict_paths["5_json_divided"], neuron_id)
        paths2process = [path_main] + [os.path.join(path_div,i) for i in os.listdir(path_div)]
        output_path = os.path.join(self.dict_paths["6-clumpiness_scores"], neuron_id)
        os.makedirs(output_path)


        for i in paths2process:
            process_single_clumpiness(filepath = i,
                                      output_dir = output_path,
                                      overwrite = self.dict_config["overwrite_info"])

        
    ##################################
    ############# STEP 7 #############
    ##################################
    def step07_result_annealing(self):
        """
        7th step, joining the clumpiness results of every neuron (step 6) into a single unified csv file.
        One row per tree: neuron_id, node_id (0 = root / main tree) and one column per label combination (null if not found).
        """

        save_path = os.path.join(self.dict_paths["7-swc_clumpiness_result"], "unified_clumpiness.csv")

        return compile_wide_clumpiness(input_directory = self.dict_paths["6-clumpiness_scores"],
                                       output_filepath = save_path,
                                       n_jobs = self.dict_config["n_jobs"])


    ##################################
    ############# STEP 8 #############
    ##################################
    def step08_score_assignment(self,
                                trees : tuple = ("simplified", "raw")):
        """
        8th step, assigning the clumpiness scores of the unified file (step 7) to the root and the internal nodes
        of each neuron's SWC file - every other node stays null. The join is based on the node id.
        Output: `8-swc_clumpiness/<simplified | raw>/<neuron_id>.csv` - the SWC nodes + one column per label combination.
        trees: tuple -> which trees to write: "simplified" (step 3 file, with the labels) and / or "raw" (input swc file).
        """

        unified_path = os.path.join(self.dict_paths["7-swc_clumpiness_result"], "unified_clumpiness.csv")

        return assign_unified_clumpiness(unified_filepath = unified_path,
                                         simplified_dir = self.dict_paths["3-swc_simplified_labeled"],
                                         raw_dir = self.dict_config["swc_path"],
                                         output_dir = self.dict_paths["8-swc_clumpiness"],
                                         trees = trees,
                                         overwrite = self.dict_config["overwrite_info"],
                                         n_jobs = self.dict_config["n_jobs"])



    ##################################
    ############# STEP 9 #############
    ##################################
    def step09_plot_creation(self,
                             trees : tuple = ("simplified", "raw")):
        """
        9th step, saving the interactive topology plot of each neuron (step 8 file) as a stand-alone html file -
        node fill = clumpiness score, shape and colour = label. See `plot_neuron_topology` in `scripts/visualize.py`.
        Output: `9-swc_clumpiness_plots/<simplified | raw>/<neuron_id>.html`, the same layout as step 8.
        trees: tuple -> which trees to plot: "simplified" and / or "raw".
        """

        return plot_scored_neurons(scored_dir = self.dict_paths["8-swc_clumpiness"],
                                   labels_dir = self.dict_paths["3-swc_simplified_labeled"],
                                   output_dir = self.dict_paths["9-swc_clumpiness_plots"],
                                   trees = trees,
                                   overwrite = self.dict_config["overwrite_info"],
                                   n_jobs = self.dict_config["n_jobs"])


In [3]:
path_div = "output\\test_pipeline\\5_json_divided\\720575940596125868"
[os.path.join(path_div,i) for i in os.listdir(path_div)]

['output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_10.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_1004.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_1014.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_1015.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_1017.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_1018.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_1019.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_1020.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_1026.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_1030.json',
 'output\\test_pipeline\\5_json_divided\\720575940596125868\\720575940596125868_10

In [4]:
# Fixed class (scripts/process_swc.py) - replaces the class above when mode == "claude"
# Fixes: step 3 labels aligned by node_id, "False"/"None" config parsing, one overwrite rule for all steps,
# labels parquet path set in __init__, every step returns stats. See _archive/BUGFIX_REPORT.md.
import importlib
import scripts.preprocessing, scripts.processing, scripts.visualize, scripts.process_swc

# Reload in dependency order -> edits to the .py files are picked up without restarting the kernel
for module in (scripts.preprocessing, scripts.processing, scripts.visualize, scripts.process_swc):
    importlib.reload(module)

ProcessSWC = scripts.process_swc.ProcessSWC

In [5]:
# Run pipeline

if __name__ == '__main__':
    test = ProcessSWC()

In [6]:
# Test run - process a selection of neurons step by step, record failures and validate each step's output
from tqdm.auto import tqdm
import traceback
import random
import glob
import json
import time

######################
### Test settings  ###
N_NEURONS  = 10          # number of neurons to test
SELECTION  = "first"    # "first" (sorted ids) | "random"
SEED       = 0          # seed for SELECTION = "random"
NEURON_IDS = []         # explicit ids -> overrides N_NEURONS / SELECTION, e.g. ["720575940596125868"]
RUN_CHECKS = True       # validate each neuron's outputs after the steps ran
OVERWRITE  = True       # None -> config `overwrite_info`; True/False -> override for steps 2-5 only (labels parquet untouched)

# Fresh instance, so changes in the class cell are picked up (re-run the class cell first)
test = ProcessSWC()

# Step 1 - labels parquet (needed by every neuron)
print(f"> {STEP_LABELS['step01']}", flush=True)
test.step01_create_labels()
labels_file = os.path.join(test.dict_paths["1-output_labels"], "swc_labels.parquet")

# Steps 2-5 overwrite override (after step 1, so the 2GB labels parquet is not rebuilt)
if OVERWRITE is not None:
    test.dict_config["overwrite_info"] = OVERWRITE

# Candidate neurons -> raw SWC file AND labels exist
swc_ids = {os.path.splitext(f)[0] for f in os.listdir(test.dict_config["swc_path"]) if f.endswith(".swc")}
labeled_ids = set(pl.scan_parquet(labels_file).select(pl.col("neuron").cast(pl.Utf8)).unique().collect()["neuron"].to_list())
candidates = sorted(swc_ids & labeled_ids)

if NEURON_IDS:
    selected = [str(i) for i in NEURON_IDS]
elif SELECTION == "random":
    selected = random.Random(SEED).sample(candidates, min(N_NEURONS, len(candidates)))
else:
    selected = candidates[:N_NEURONS]

print(f"> {len(swc_ids)} SWC files, {len(candidates)} with labels, testing {len(selected)} (overwrite={test.dict_config['overwrite_info']}).")


######################
### Per-step checks ###
def check_neuron(neuron_id: str) -> dict:
    """Independent checks on the files written by steps 2-5 (does not use the pipeline functions)."""
    checks = {}

    # Step 2 - simplified tree has exactly one root
    swc_simp = pd.read_csv(os.path.join(test.dict_paths["2-swc_simplified"], f"{neuron_id}.csv"), index_col=0)
    checks["n_nodes_simplified"] = len(swc_simp)
    checks["ok_single_root"] = int((swc_simp["parent"] == -1).sum()) == 1

    # Step 3 - one row per node, and every node carries exactly its own labels
    swc_lab = pd.read_csv(os.path.join(test.dict_paths["3-swc_simplified_labeled"], f"{neuron_id}.csv"), index_col=0)
    checks["ok_unique_nodes"] = swc_lab["node_id"].is_unique

    labels = pl.scan_parquet(labels_file).filter(pl.col("neuron").cast(pl.Utf8) == neuron_id).select(["node_id", "type"]).collect().to_pandas()
    truth = labels.dropna().groupby("node_id")["type"].agg(lambda s: set(s.astype(str)))
    expected = swc_lab["node_id"].map(truth).apply(lambda v: v if isinstance(v, set) else set())
    found = swc_lab["type"].apply(lambda v: set(str(v).split(",")) if pd.notna(v) else set())
    checks["labeled_nodes_raw"] = labels["node_id"].nunique()
    checks["labeled_nodes_expected"] = int((expected.apply(len) > 0).sum())
    checks["labeled_nodes_found"] = int((found.apply(len) > 0).sum())
    checks["label_mismatches"] = int((expected != found).sum())
    checks["ok_labels"] = checks["label_mismatches"] == 0

    # Step 4 - every csv node is in the json tree, with the same labels as the step 3 csv
    with open(os.path.join(test.dict_paths["4-json"], f"{neuron_id}.json")) as f:
        tree = json.load(f)
    json_labels, stack = {}, [tree]
    while stack:
        node = stack.pop()
        json_labels[node[0]["nodeID"]] = set(node[0]["nodeLabels"])
        stack.extend(node[1])
    csv_labels = {str(int(n)): {x.strip().lower() for x in labs} for n, labs in zip(swc_lab["node_id"], found)}
    checks["ok_json_nodes"] = len(json_labels) == len(swc_lab)
    checks["ok_json_labels"] = json_labels == csv_labels

    # Step 5 - one sub-tree per internal node, root excluded
    n_internal = int(swc_lab["node_id"].isin(swc_lab["parent"]).sum())
    n_subtrees = len(glob.glob(os.path.join(test.dict_paths["5_json_divided"], neuron_id, "*.json")))
    checks["ok_subtrees"] = n_subtrees == n_internal - 1

    # Step 6 - one clumpiness csv per json tree (main tree + every sub-tree)
    n_csv = len(glob.glob(os.path.join(test.dict_paths["6-clumpiness_scores"], neuron_id, "*.csv")))
    checks["ok_clumpiness"] = n_csv == n_subtrees + 1

    return checks


######################
### Run            ###
steps = [("step02", test.step02_simplify_swc),
         ("step03", test.step03_label_joining),
         ("step04", test.step04_json_creation),
         ("step05", test.step05_internal_node_div),
         ("step06", test.step06_clumpiness_calculation)]

records = {neuron_id: {"neuron": neuron_id, "status": "ok", "failed_step": None, "error": None} for neuron_id in selected}
tracebacks = {}

# One step at a time over all neurons -> every step announces itself and has its own progress bar.
# A neuron that failed a step is left out of the next steps.
for step_name, step_fn in steps:
    active = [neuron_id for neuron_id in selected if records[neuron_id]["status"] == "ok"]
    print(f"> {STEP_LABELS[step_name]} ({len(active)} neurons)", flush=True)
    for neuron_id in tqdm(active, desc=STEP_LABELS[step_name].split(" - ")[0]):
        t0 = time.time()
        try:
            step_fn(neuron_id)
        except Exception as err:
            records[neuron_id].update(status="failed", failed_step=step_name, error=f"{type(err).__name__}: {err}")
            tracebacks[neuron_id] = traceback.format_exc()
            continue
        records[neuron_id][f"{step_name}_sec"] = round(time.time() - t0, 2)

if RUN_CHECKS:
    active = [neuron_id for neuron_id in selected if records[neuron_id]["status"] == "ok"]
    print(f"> Checks - validating the output files ({len(active)} neurons)", flush=True)
    for neuron_id in tqdm(active, desc="Checks"):
        try:
            records[neuron_id].update(check_neuron(neuron_id))
        except Exception as err:
            records[neuron_id].update(status="check_error", error=f"{type(err).__name__}: {err}")
            tracebacks[neuron_id] = traceback.format_exc()

df_test = pd.DataFrame(list(records.values()))


######################
### Summary        ###
print(df_test["status"].value_counts().to_string())
check_cols = [c for c in df_test.columns if c.startswith("ok_")]
if check_cols:
    print("\n> Share of neurons passing each check:")
    print(df_test[check_cols].astype(float).mean().round(3).to_string())
if tracebacks:
    print(f"\n> {len(tracebacks)} neurons with errors - full traces in `tracebacks[neuron_id]`.")


######################
### Steps 7-9      ###
# Run once on every processed neuron: unified csv -> scores on the SWC nodes -> html plots
run_steps = time_run_steps([("step07", test.step07_result_annealing, os.path.join(test.dict_paths["7-swc_clumpiness_result"], "unified_clumpiness.csv")),
                            ("step08", test.step08_score_assignment, test.dict_paths["8-swc_clumpiness"]),
                            ("step09", test.step09_plot_creation, test.dict_paths["9-swc_clumpiness_plots"])])


######################
### Report         ###
# html report of this run -> `0-run_log` folder, with links to the plots of step 9
report_path = write_test_report(df_test, tracebacks,
                                out_dir   = test.dict_paths["0-run_log"],
                                run_name  = test.dict_config["run_name"],
                                settings  = {"SWC files": len(swc_ids), "with labels": len(candidates), "tested": len(selected),
                                             "selection": "explicit ids" if NEURON_IDS else SELECTION, "seed": SEED,
                                             "run checks": RUN_CHECKS, "overwrite": test.dict_config["overwrite_info"]},
                                run_steps = run_steps,
                                plots_dir = test.dict_paths["9-swc_clumpiness_plots"])
print(f"> Report saved to {report_path}")

df_test

c:\Users\Daniel\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


> Function execution halted, old `swc_labels.parquet` file preserved.
> 200 SWC files, 178 with labels, testing 10 (overwrite=True).


Testing neurons: 100%|██████████| 10/10 [00:58<00:00,  5.83s/it]

status
failed    10

> 10 neurons with errors - full traces in `tracebacks[neuron_id]`.


,neuron,status,failed_step,error,step02_sec,step03_sec,step04_sec,step05_sec
0,720575940596125868,failed,step06,Exception: > Failed to process clumpiness on `...,1.82,1.00,0.07,2.05
1,720575940597856265,failed,step06,Exception: > Failed to process clumpiness on `...,1.02,1.00,0.14,7.55
2,720575940597944841,failed,step06,Exception: > Failed to process clumpiness on `...,0.98,0.95,0.08,3.39
3,720575940598267657,failed,step06,Exception: > Failed to process clumpiness on `...,0.98,0.96,0.19,10.58
4,720575940599333574,failed,step06,Exception: > Failed to process clumpiness on `...,0.88,0.93,0.08,4.57
5,720575940599457990,failed,step06,Exception: > Failed to process clumpiness on `...,0.97,0.90,0.04,1.42
6,720575940599459782,failed,step06,Exception: > Failed to process clumpiness on `...,0.95,0.91,0.04,1.38
7,720575940599704006,failed,step06,Exception: > Failed to process clumpiness on `...,0.96,0.92,0.05,2.01
8,720575940599729862,failed,step06,Exception: > Failed to process clumpiness on `...,0.96,0.96,0.06,2.30
9,720575940599733446,failed,step06,Exception: > Failed to process clumpiness on `...,0.99,0.94,0.06,1.98
